# Dataset Audit (CSV)

This notebook is **only** for inspecting and validating the raw dataset format. It does **not** train models.

## What you should learn after running
- Are rows **wide** (one row per scenario) or **long** (many rows per scenario over time)?
- What are the **sensor pressure** columns?
- What are the **target** columns for leak x, leak y, and burst size?
- Is there a **scenario_id** column?
- Are scenarios exactly **24 hours** long (if long format)?

This notebook writes a metadata summary to `outputs/metadata/`.

In [1]:
from __future__ import annotations

import json
import re
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

print('pandas:', pd.__version__)
print('numpy:', np.__version__)

pandas: 3.0.3
numpy: 2.4.6


In [2]:
# ---- Configuration ----
# Option A: set your CSV path directly (recommended):
CSV_PATH = "data/raw/leak_dataset_wide.csv"  # wide format raw dataset

# Option B: auto-detect a CSV under these folders if CSV_PATH is None:
SEARCH_DIRS = [Path('data/raw'), Path('data'), Path('.')]

# Scenario identifier (explicit)
SCENARIO_ID_COL = "scenario_id"

# Sensor columns are named like: NODEADD_2423_Hour2 / NODE_12_Hour24 / HOUSE_EPN_7_Hour1
# Provide allowed node-type prefixes here (case-insensitive):
SENSOR_NODE_PREFIXES = ["NODEADD", "NODE", "HOUSE_EPN"]

# If your sensor columns share some *other* prefix (e.g. pressure_), you can set it here; otherwise keep None:
SENSOR_PREFIX_HINT = None

# Explicit targets (as per dataset):
TARGET_X_COL = "leak_x"
TARGET_Y_COL = "leak_y"
TARGET_BURST_COL = "leak_size_lpm"

# Targets (used as fallback name-based suggestions too):
TARGET_HINTS = [TARGET_X_COL, TARGET_Y_COL, TARGET_BURST_COL, "x_coord", "y_coord", "burst_size"]

# Scenario/time hints (used as fallback if SCENARIO_ID_COL is missing):
SCENARIO_ID_HINTS = ["scenario_id", "scenario", "case_id", "run_id", "id", "leak"]
HOUR_HINTS = ["hour", "hr"]
TIMESTAMP_HINTS = ["timestamp", "datetime", "date"]

# For large CSVs, you can set SAMPLE_ROWS to a number (e.g. 200_000).
SAMPLE_ROWS = None

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)

In [3]:
def discover_project_root(start: Path) -> Path:
    """Walk upward until we find a marker of the repo root."""
    start = start.resolve()
    for p in [start, *start.parents]:
        if (p / ".git").exists() or (p / "README.md").exists():
            return p
    return start


PROJECT_ROOT = discover_project_root(Path.cwd())
print("Working directory:", Path.cwd())
print("Detected PROJECT_ROOT:", PROJECT_ROOT)


def resolve_path_maybe_from_root(p: Path) -> Path:
    """Resolve a possibly-relative path against the detected project root."""
    p = Path(p).expanduser()
    if p.is_absolute():
        return p.resolve()
    # Try relative to current working directory first
    if p.exists():
        return p.resolve()
    # Fall back to project root
    return (PROJECT_ROOT / p).resolve()


def find_candidate_csvs(search_dirs: List[Path]) -> List[Path]:
    candidates: List[Path] = []
    for d in search_dirs:
        d2 = resolve_path_maybe_from_root(d)
        if d2.exists() and d2.is_dir():
            candidates.extend(sorted(d2.rglob("*.csv")))
    # De-dup while preserving order
    seen: set[Path] = set()
    unique: List[Path] = []
    for p in candidates:
        rp = p.resolve()
        if rp not in seen:
            seen.add(rp)
            unique.append(p)
    return unique


def pick_csv_path(csv_path: Optional[str]) -> Path:
    if csv_path:
        p_raw = Path(csv_path).expanduser()
        # Absolute or relative-to-cwd
        p1 = resolve_path_maybe_from_root(p_raw)
        if not p1.exists():
            raise FileNotFoundError(
                "CSV_PATH points to a missing file. "
                f"Tried: {p_raw} (cwd={Path.cwd()}) and {p1} (PROJECT_ROOT={PROJECT_ROOT})"
            )
        return p1

    candidates = find_candidate_csvs(SEARCH_DIRS)
    if not candidates:
        raise FileNotFoundError("No CSV files found. Set CSV_PATH or put a CSV under data/raw/.")

    print("Detected CSV candidates:")
    for i, p in enumerate(candidates, start=1):
        print(f"  {i:>2}. {p.as_posix()}")

    # Choose the first by default; change idx if needed.
    idx = 1
    chosen = candidates[idx - 1]
    print()
    print(f"Using: {chosen.as_posix()}")
    return chosen


def load_csv(path: Path, sample_rows: Optional[int] = None) -> pd.DataFrame:
    read_kwargs: Dict[str, Any] = {"low_memory": False}
    if sample_rows is not None:
        read_kwargs["nrows"] = int(sample_rows)

    try:
        return pd.read_csv(path, **read_kwargs)
    except Exception as e:
        print(f"Primary read_csv failed: {e!r}")
        return pd.read_csv(path, engine="python", **read_kwargs)


csv_path = pick_csv_path(CSV_PATH)
df = load_csv(csv_path, sample_rows=SAMPLE_ROWS)
df.head()

Working directory: /home/sujalmainali727/Projects/leak-detection-models/notebooks
Detected PROJECT_ROOT: /home/sujalmainali727/Projects/leak-detection-models


,scenario_id,leak,leak_node,leak_zone,leak_zone_multiplier,leak_node_effective_weight,leak_x,leak_y,leak_start_hr,leak_duration_hr,emitter_coeff_baseline,emitter_coeff_added,emitter_coeff_total,leak_size_lpm,leak_node_pressure,leak_magnitude_class,validation_status,NODEADD_2423_Hour0,NODEADD_2423_Hour1,NODEADD_2423_Hour2,NODEADD_2423_Hour3,NODEADD_2423_Hour4,NODEADD_2423_Hour5,NODEADD_2423_Hour6,NODEADD_2423_Hour7,NODEADD_2423_Hour8,NODEADD_2423_Hour9,NODEADD_2423_Hour10,NODEADD_2423_Hour11,NODEADD_2423_Hour12,NODEADD_2423_Hour13,NODEADD_2423_Hour14,NODEADD_2423_Hour15,NODEADD_2423_Hour16,NODEADD_2423_Hour17,NODEADD_2423_Hour18,NODEADD_2423_Hour19,NODEADD_2423_Hour20,NODEADD_2423_Hour21,NODEADD_2423_Hour22,NODEADD_2423_Hour23,NODE_3005_Hour0,NODE_3005_Hour1,NODE_3005_Hour2,NODE_3005_Hour3,NODE_3005_Hour4,NODE_3005_Hour5,NODE_3005_Hour6,NODE_3005_Hour7,NODE_3005_Hour8,NODE_3005_Hour9,NODE_3005_Hour10,NODE_3005_Hour11,NODE_3005_Hour12,NODE_3005_Hour13,NODE_3005_Hour14,NODE_3005_Hour15,NODE_3005_Hour16,NODE_3005_Hour17,NODE_3005_Hour18,NODE_3005_Hour19,NODE_3005_Hour20,NODE_3005_Hour21,NODE_3005_Hour22,NODE_3005_Hour23,HOUSE_EPN_164_Hour0,HOUSE_EPN_164_Hour1,HOUSE_EPN_164_Hour2,HOUSE_EPN_164_Hour3,HOUSE_EPN_164_Hour4,HOUSE_EPN_164_Hour5,HOUSE_EPN_164_Hour6,HOUSE_EPN_164_Hour7,HOUSE_EPN_164_Hour8,HOUSE_EPN_164_Hour9,HOUSE_EPN_164_Hour10,HOUSE_EPN_164_Hour11,HOUSE_EPN_164_Hour12,HOUSE_EPN_164_Hour13,HOUSE_EPN_164_Hour14,HOUSE_EPN_164_Hour15,HOUSE_EPN_164_Hour16,HOUSE_EPN_164_Hour17,HOUSE_EPN_164_Hour18,HOUSE_EPN_164_Hour19,HOUSE_EPN_164_Hour20,HOUSE_EPN_164_Hour21,HOUSE_EPN_164_Hour22,HOUSE_EPN_164_Hour23,NODEADD_022_Hour0,NODEADD_022_Hour1,NODEADD_022_Hour2,NODEADD_022_Hour3,NODEADD_022_Hour4,NODEADD_022_Hour5,NODEADD_022_Hour6,NODEADD_022_Hour7,NODEADD_022_Hour8,NODEADD_022_Hour9,NODEADD_022_Hour10,...,NODE_3116_Hour20,NODE_3116_Hour21,NODE_3116_Hour22,NODE_3116_Hour23,HOUSE_EPN_695_Hour0,HOUSE_EPN_695_Hour1,HOUSE_EPN_695_Hour2,HOUSE_EPN_695_Hour3,HOUSE_EPN_695_Hour4,HOUSE_EPN_695_Hour5,HOUSE_EPN_695_Hour6,HOUSE_EPN_695_Hour7,HOUSE_EPN_695_Hour8,HOUSE_EPN_695_Hour9,HOUSE_EPN_695_Hour10,HOUSE_EPN_695_Hour11,HOUSE_EPN_695_Hour12,HOUSE_EPN_695_Hour13,HOUSE_EPN_695_Hour14,HOUSE_EPN_695_Hour15,HOUSE_EPN_695_Hour16,HOUSE_EPN_695_Hour17,HOUSE_EPN_695_Hour18,HOUSE_EPN_695_Hour19,HOUSE_EPN_695_Hour20,HOUSE_EPN_695_Hour21,HOUSE_EPN_695_Hour22,HOUSE_EPN_695_Hour23,NODE_3094_Hour0,NODE_3094_Hour1,NODE_3094_Hour2,NODE_3094_Hour3,NODE_3094_Hour4,NODE_3094_Hour5,NODE_3094_Hour6,NODE_3094_Hour7,NODE_3094_Hour8,NODE_3094_Hour9,NODE_3094_Hour10,NODE_3094_Hour11,NODE_3094_Hour12,NODE_3094_Hour13,NODE_3094_Hour14,NODE_3094_Hour15,NODE_3094_Hour16,NODE_3094_Hour17,NODE_3094_Hour18,NODE_3094_Hour19,NODE_3094_Hour20,NODE_3094_Hour21,NODE_3094_Hour22,NODE_3094_Hour23,HOUSE_EPN_638_Hour0,HOUSE_EPN_638_Hour1,HOUSE_EPN_638_Hour2,HOUSE_EPN_638_Hour3,HOUSE_EPN_638_Hour4,HOUSE_EPN_638_Hour5,HOUSE_EPN_638_Hour6,HOUSE_EPN_638_Hour7,HOUSE_EPN_638_Hour8,HOUSE_EPN_638_Hour9,HOUSE_EPN_638_Hour10,HOUSE_EPN_638_Hour11,HOUSE_EPN_638_Hour12,HOUSE_EPN_638_Hour13,HOUSE_EPN_638_Hour14,HOUSE_EPN_638_Hour15,HOUSE_EPN_638_Hour16,HOUSE_EPN_638_Hour17,HOUSE_EPN_638_Hour18,HOUSE_EPN_638_Hour19,HOUSE_EPN_638_Hour20,HOUSE_EPN_638_Hour21,HOUSE_EPN_638_Hour22,HOUSE_EPN_638_Hour23,NODEADD_1830_Hour0,NODEADD_1830_Hour1,NODEADD_1830_Hour2,NODEADD_1830_Hour3,NODEADD_1830_Hour4,NODEADD_1830_Hour5,NODEADD_1830_Hour6,NODEADD_1830_Hour7,NODEADD_1830_Hour8,NODEADD_1830_Hour9,NODEADD_1830_Hour10,NODEADD_1830_Hour11,NODEADD_1830_Hour12,NODEADD_1830_Hour13,NODEADD_1830_Hour14,NODEADD_1830_Hour15,NODEADD_1830_Hour16,NODEADD_1830_Hour17,NODEADD_1830_Hour18,NODEADD_1830_Hour19,NODEADD_1830_Hour20,NODEADD_1830_Hour21,NODEADD_1830_Hour22,NODEADD_1830_Hour23
0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,success,21.806149,21.803417,21.783144,21.686366,21.379463,20.813671,20.467462,20.816489,21.381579,21.687092,21.782953,21.794518,21.695739,21.210188,20.685352,21.210998,21.696181,21.79

In [4]:
print('CSV:', csv_path.as_posix())
print('Shape:', df.shape)
print('Columns:', len(df.columns))

pd.Series(df.columns, name='columns')

CSV: /home/sujalmainali727/Projects/leak-detection-models/data/raw/leak_dataset_wide.csv
Shape: (8001, 521)
Columns: 521


0               scenario_id
1                      leak
2                 leak_node
3                 leak_zone
4      leak_zone_multiplier
               ...         
516     NODEADD_1830_Hour19
517     NODEADD_1830_Hour20
518     NODEADD_1830_Hour21
519     NODEADD_1830_Hour22
520     NODEADD_1830_Hour23
Name: columns, Length: 521, dtype: str

In [18]:
# Data types overview
dtype_summary = df.dtypes.astype(str).value_counts().rename_axis('dtype').reset_index(name='count')
dtype_summary

,dtype,count
0,float64,515
1,str,4
2,int64,2


In [5]:
# Missing values overview
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (missing / max(len(df), 1)) * 100
missing_table = pd.DataFrame({'missing': missing, 'missing_pct': missing_pct}).query('missing > 0')
missing_table.head(50)

,missing,missing_pct
leak_x,1,0.012498
leak_start_hr,1,0.012498
leak_y,1,0.012498
leak_magnitude_class,1,0.012498
leak_node,1,0.012498
leak_zone,1,0.012498
leak_zone_multiplier,1,0.012498
leak_node_effective_weight,1,0.012498
leak_duration_hr,1,0.012498
emitter_coeff_baseline,1,0.012498


In [20]:
# Missing values: explicit summary for baseline decisions
target_cols = [TARGET_X_COL, TARGET_Y_COL, TARGET_BURST_COL]
target_present = [c for c in target_cols if c in df.columns]
sensor_present = [c for c in sensor_cols if c in df.columns]

missing_by_col = df.isna().sum().sort_values(ascending=False)
missing_total = int(missing_by_col.sum())
print("Total missing cells:", missing_total)

if target_present:
    print("Missing in targets:")
    for c in target_present:
        print(f" - {c}: {int(df[c].isna().sum())}")
    rows_missing_any_target = int(df[target_present].isna().any(axis=1).sum())
    print("Rows missing ANY target:", rows_missing_any_target)

rows_missing_any_sensor = int(df[sensor_present].isna().any(axis=1).sum()) if sensor_present else 0
print("Rows missing ANY sensor feature:", rows_missing_any_sensor)

rows_missing_any_baseline = int(df[target_present + sensor_present].isna().any(axis=1).sum()) if (target_present or sensor_present) else 0
print("Rows missing ANY baseline field (targets + sensors):", rows_missing_any_baseline)

print()
print("Top columns by missing count:")
display(missing_by_col.head(25).rename("missing"))

Total missing cells: 14
Missing in targets:
 - leak_x: 1
 - leak_y: 1
 - leak_size_lpm: 1
Rows missing ANY target: 1
Rows missing ANY sensor feature: 0
Rows missing ANY baseline field (targets + sensors): 1

Top columns by missing count:


leak_x                        1
leak_start_hr                 1
leak_y                        1
leak_magnitude_class          1
leak_node                     1
leak_zone                     1
leak_zone_multiplier          1
leak_node_effective_weight    1
leak_duration_hr              1
emitter_coeff_baseline        1
emitter_coeff_added           1
emitter_coeff_total           1
leak_size_lpm                 1
leak_node_pressure            1
NODE_3028_Hour22              0
NODE_3028_Hour21              0
NODE_3028_Hour20              0
NODE_3028_Hour19              0
NODE_3028_Hour17              0
NODE_3028_Hour23              0
NODE_3024_Hour0               0
NODE_3024_Hour1               0
NODE_3024_Hour2               0
NODE_3024_Hour3               0
NODE_3024_Hour4               0
Name: missing, dtype: int64

In [6]:
# Duplicate rows check
dup_count = int(df.duplicated().sum())
print('Duplicated rows:', dup_count)
df[df.duplicated()].head() if dup_count else 'No duplicate rows detected.'

Duplicated rows: 0


'No duplicate rows detected.'

In [10]:
def normalize_col(c: str) -> str:
    return re.sub(r'\\s+', '_', str(c).strip().lower())


def contains_any(col: str, hints: List[str]) -> bool:
    c = normalize_col(col)
    return any(h in c for h in hints)


numeric_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
non_numeric_cols = [c for c in df.columns if c not in numeric_cols]

scenario_candidates = [c for c in df.columns if contains_any(c, SCENARIO_ID_HINTS)]
hour_candidates = [c for c in df.columns if contains_any(c, HOUR_HINTS)]
timestamp_candidates = [c for c in df.columns if contains_any(c, TIMESTAMP_HINTS)]
target_name_candidates = [c for c in df.columns if contains_any(c, TARGET_HINTS)]

print('Numeric cols:', len(numeric_cols))
print('Non-numeric cols:', len(non_numeric_cols))
print()
print('Scenario ID candidates:', scenario_candidates)
print('Hour-like candidates:', hour_candidates)
print('Timestamp-like candidates:', timestamp_candidates)
print('Target-name candidates:', target_name_candidates)

Numeric cols: 517
Non-numeric cols: 4

Scenario ID candidates: ['scenario_id', 'leak', 'leak_node', 'leak_zone', 'leak_zone_multiplier', 'leak_node_effective_weight', 'leak_x', 'leak_y', 'leak_start_hr', 'leak_duration_hr', 'leak_size_lpm', 'leak_node_pressure', 'leak_magnitude_class', 'validation_status']
Hour-like candidates: ['leak_start_hr', 'leak_duration_hr', 'NODEADD_2423_Hour0', 'NODEADD_2423_Hour1', 'NODEADD_2423_Hour2', 'NODEADD_2423_Hour3', 'NODEADD_2423_Hour4', 'NODEADD_2423_Hour5', 'NODEADD_2423_Hour6', 'NODEADD_2423_Hour7', 'NODEADD_2423_Hour8', 'NODEADD_2423_Hour9', 'NODEADD_2423_Hour10', 'NODEADD_2423_Hour11', 'NODEADD_2423_Hour12', 'NODEADD_2423_Hour13', 'NODEADD_2423_Hour14', 'NODEADD_2423_Hour15', 'NODEADD_2423_Hour16', 'NODEADD_2423_Hour17', 'NODEADD_2423_Hour18', 'NODEADD_2423_Hour19', 'NODEADD_2423_Hour20', 'NODEADD_2423_Hour21', 'NODEADD_2423_Hour22', 'NODEADD_2423_Hour23', 'NODE_3005_Hour0', 'NODE_3005_Hour1', 'NODE_3005_Hour2', 'NODE_3005_Hour3', 'NODE_3005_Hou

In [11]:
def parse_sensor_column_name(col: str, allowed_prefixes: List[str]) -> Optional[Dict[str, Any]]:
    """Parse names like NODEADD_12_HOUR2 into {node_type, node_id, hour}."""
    cn = normalize_col(col)
    prefixes_norm = [normalize_col(p) for p in allowed_prefixes]
    prefix_group = "|".join(map(re.escape, prefixes_norm)) if prefixes_norm else ""
    if not prefix_group:
        return None
    m = re.match(rf"^(?P<node_type>{prefix_group})_(?P<node_id>[a-z0-9]+)_hour(?P<hour>\d{{1,3}})$", cn)
    if not m:
        return None
    hour = int(m.group("hour"))
    return {
        "node_type": m.group("node_type"),
        "node_id": m.group("node_id"),
        "hour": hour,
    }


def detect_sensor_columns(
    df: pd.DataFrame,
    *,
    prefix_hint: Optional[str] = None,
    node_prefixes: Optional[List[str]] = None,
    valid_hours: Optional[range] = range(1, 25),
    require_numeric: bool = True,
    ) -> Tuple[List[str], pd.DataFrame]:
    """Return (sensor_cols, sensor_info_df) based on name parsing and dtype checks."""
    cols = list(df.columns)
    if require_numeric:
        cols = [c for c in cols if pd.api.types.is_numeric_dtype(df[c])]

    # 1) If prefix_hint is set and matches numeric columns, treat those as sensors
    if prefix_hint:
        ph = normalize_col(prefix_hint)
        hinted = [c for c in cols if normalize_col(c).startswith(ph)]
        if hinted:
            info = [{"column": c} for c in hinted]
            return hinted, pd.DataFrame(info)

    # 2) Preferred: parse NODEADD/NODE/HOUSE_EPN patterns
    node_prefixes = node_prefixes or []
    parsed: List[Dict[str, Any]] = []
    for c in cols:
        out = parse_sensor_column_name(c, node_prefixes)
        if out is None:
            continue
        if valid_hours is not None and out["hour"] not in valid_hours:
            # if dataset uses 0-23 or other scheme, disable/adjust valid_hours in config
            continue
        parsed.append({"column": c, **out})

    if parsed:
        info_df = pd.DataFrame(parsed).sort_values(["node_type", "node_id", "hour"])
        return info_df["column"].tolist(), info_df

    # 3) Fallback heuristic: numeric columns with digits and underscores
    sensor_like: List[str] = []
    for c in cols:
        cn = normalize_col(c)
        digit_count = sum(ch.isdigit() for ch in cn)
        underscore_count = cn.count("_")
        if digit_count >= 1 and underscore_count >= 1:
            sensor_like.append(c)
    info_df = pd.DataFrame([{"column": c} for c in sensor_like])
    return (sensor_like if sensor_like else cols), info_df


sensor_cols, sensor_info = detect_sensor_columns(
    df,
    prefix_hint=SENSOR_PREFIX_HINT,
    node_prefixes=SENSOR_NODE_PREFIXES,
    valid_hours=range(0, 24),
    require_numeric=True,
 )

print("Detected sensor-like columns:", len(sensor_cols))
display(pd.Series(sensor_cols[:80], name="sensor_cols_sample"))

if not sensor_info.empty and {"node_type", "node_id", "hour"}.issubset(sensor_info.columns):
    print()
    print("Sensor naming parsed successfully.")
    display(sensor_info.head(10))
    print()
    print("Node types:", sensor_info["node_type"].value_counts().to_dict())
    print("Unique nodes:", int(sensor_info[["node_type", "node_id"]].drop_duplicates().shape[0]))
    print("Hour coverage (min/max/distinct):", int(sensor_info["hour"].min()), int(sensor_info["hour"].max()), int(sensor_info["hour"].nunique()))
    missing_hours = sorted(set(range(0, 24)) - set(sensor_info["hour"].unique().tolist()))
    print("Missing hours (0-23):", missing_hours[:50] if missing_hours else "None")

Detected sensor-like columns: 504


0      HOUSE_EPN_164_Hour0
1      HOUSE_EPN_164_Hour1
2      HOUSE_EPN_164_Hour2
3      HOUSE_EPN_164_Hour3
4      HOUSE_EPN_164_Hour4
5      HOUSE_EPN_164_Hour5
6      HOUSE_EPN_164_Hour6
7      HOUSE_EPN_164_Hour7
8      HOUSE_EPN_164_Hour8
9      HOUSE_EPN_164_Hour9
10    HOUSE_EPN_164_Hour10
11    HOUSE_EPN_164_Hour11
12    HOUSE_EPN_164_Hour12
13    HOUSE_EPN_164_Hour13
14    HOUSE_EPN_164_Hour14
15    HOUSE_EPN_164_Hour15
16    HOUSE_EPN_164_Hour16
17    HOUSE_EPN_164_Hour17
18    HOUSE_EPN_164_Hour18
19    HOUSE_EPN_164_Hour19
20    HOUSE_EPN_164_Hour20
21    HOUSE_EPN_164_Hour21
22    HOUSE_EPN_164_Hour22
23    HOUSE_EPN_164_Hour23
24     HOUSE_EPN_255_Hour0
25     HOUSE_EPN_255_Hour1
26     HOUSE_EPN_255_Hour2
27     HOUSE_EPN_255_Hour3
28     HOUSE_EPN_255_Hour4
29     HOUSE_EPN_255_Hour5
30     HOUSE_EPN_255_Hour6
31     HOUSE_EPN_255_Hour7
32     HOUSE_EPN_255_Hour8
33     HOUSE_EPN_255_Hour9
34    HOUSE_EPN_255_Hour10
35    HOUSE_EPN_255_Hour11
36    HOUSE_EPN_255_Hour12
3


Sensor naming parsed successfully.


,column,node_type,node_id,hour
48,HOUSE_EPN_164_Hour0,house_epn,164,0
49,HOUSE_EPN_164_Hour1,house_epn,164,1
50,HOUSE_EPN_164_Hour2,house_epn,164,2
51,HOUSE_EPN_164_Hour3,house_epn,164,3
52,HOUSE_EPN_164_Hour4,house_epn,164,4
53,HOUSE_EPN_164_Hour5,house_epn,164,5
54,HOUSE_EPN_164_Hour6,house_epn,164,6
55,HOUSE_EPN_164_Hour7,house_epn,164,7
56,HOUSE_EPN_164_Hour8,house_epn,164,8
57,HOUSE_EPN_164_Hour9,house_epn,164,9



Node types: {'node': 288, 'house_epn': 144, 'nodeadd': 72}
Unique nodes: 21
Hour coverage (min/max/distinct): 0 23 24
Missing hours (0-23): None


In [12]:
# Sensor column ordering + 24-hour completeness (Phase 1 checklist)
import math

sensor_set = set(sensor_cols)
sensor_cols_in_df_order = [c for c in df.columns if c in sensor_set]
print("Sensor cols (df order):", len(sensor_cols_in_df_order))
print("Sensor cols (detected list):", len(sensor_cols))
if len(sensor_cols_in_df_order) != len(sensor_cols):
    missing_from_df = sorted(set(sensor_cols) - set(sensor_cols_in_df_order))
    print("WARNING: Some detected sensor cols were not found in df order scan:")
    print(missing_from_df[:50])

# If we parsed (node_type, node_id, hour), check whether each node has full 0-23 hours
if "sensor_info" in globals() and not sensor_info.empty and {"node_type", "node_id", "hour"}.issubset(sensor_info.columns):
    expected_hours = set(range(0, 24))
    per_node_hours = (
        sensor_info.groupby(["node_type", "node_id"])
        .agg(hours=("hour", lambda s: set(map(int, s.values.tolist()))))
        .reset_index()
    )
    per_node_hours["missing_hours"] = per_node_hours["hours"].apply(lambda s: sorted(expected_hours - s))
    num_nodes_incomplete = int((per_node_hours["missing_hours"].str.len() > 0).sum())
    print()
    print("Nodes with incomplete 0-23 coverage:", num_nodes_incomplete, "/", int(per_node_hours.shape[0]))
    if num_nodes_incomplete:
        display(per_node_hours.loc[per_node_hours["missing_hours"].str.len() > 0, ["node_type", "node_id", "missing_hours"]].head(20))

    # Check ordering consistency: are sensor columns already grouped by node then hour?
    expected_order = (
        sensor_info.sort_values(["node_type", "node_id", "hour"])["column"].tolist()
    )
    is_already_consistent = sensor_cols_in_df_order == expected_order
    print()
    print("Sensor columns already in consistent (node_type,node_id,hour) order:", bool(is_already_consistent))
    if not is_already_consistent:
        # show first mismatch index to help debugging
        first_mismatch = next((i for i, (a, b) in enumerate(zip(sensor_cols_in_df_order, expected_order)) if a != b), None)
        print("First mismatch index:", first_mismatch)
        if first_mismatch is not None:
            print("df column:", sensor_cols_in_df_order[first_mismatch])
            print("expected:", expected_order[first_mismatch])
else:
    print()
    print("Sensor name parsing did not yield node/hour fields; skipping per-node completeness + ordering checks.")

Sensor cols (df order): 504
Sensor cols (detected list): 504

Nodes with incomplete 0-23 coverage: 0 / 21

Sensor columns already in consistent (node_type,node_id,hour) order: False
First mismatch index: 0
df column: NODEADD_2423_Hour0
expected: HOUSE_EPN_164_Hour0


In [15]:
def pick_best_id_column(df: pd.DataFrame, candidates: List[str]) -> Optional[str]:
    if not candidates:
        return None
    best: Optional[str] = None
    best_score = -1.0
    for c in candidates:
        nun = df[c].nunique(dropna=False)
        score = nun / max(len(df), 1)
        # Prefer reasonably unique IDs but not fully unique per row
        if 0.01 <= score <= 0.99 and score > best_score:
            best, best_score = c, score
    return best or candidates[0]


sensor_set = set(sensor_cols) if "sensor_cols" in globals() else set()

if "SCENARIO_ID_COL" in globals() and SCENARIO_ID_COL in df.columns:
    scenario_id_col = SCENARIO_ID_COL
else:
    # avoid accidentally picking a sensor column as an ID
    scenario_id_col = pick_best_id_column(df, [c for c in scenario_candidates if c not in sensor_set])

# For LONG format we expect a dedicated hour/timestamp column;
# exclude sensor columns like NODE_..._Hour7 from consideration.
hour_candidates_filtered = [c for c in hour_candidates if c not in sensor_set]
timestamp_candidates_filtered = [c for c in timestamp_candidates if c not in sensor_set]

hour_col = pick_best_id_column(df, hour_candidates_filtered)
timestamp_col = pick_best_id_column(df, timestamp_candidates_filtered)

print('Chosen scenario_id_col:', scenario_id_col)
print('Chosen hour_col:', hour_col)
print('Chosen timestamp_col:', timestamp_col)

if scenario_id_col is not None:
    counts = df.groupby(scenario_id_col).size().rename('rows_per_scenario')
    display(counts.describe())
    display(counts.value_counts().head(20).rename_axis('rows_per_scenario').reset_index(name='num_scenarios'))
else:
    print('No scenario id column detected (tune SCENARIO_ID_COL/SCENARIO_ID_HINTS if needed).')

Chosen scenario_id_col: scenario_id
Chosen hour_col: leak_start_hr
Chosen timestamp_col: None


count    8001.0
mean        1.0
std         0.0
min         1.0
25%         1.0
50%         1.0
75%         1.0
max         1.0
Name: rows_per_scenario, dtype: float64

,rows_per_scenario,num_scenarios
0,1,8001


In [16]:
# WIDE vs LONG hints
# - LONG often has multiple rows per scenario_id + an hour/timestamp column.
# - WIDE often encodes hour in column names (e.g., NODEADD_12_Hour2).

def extract_hour_from_colname(col: str) -> Optional[int]:
    cn = normalize_col(col)
    m = re.search(r"(?:^|_)hour(?P<h>\d{1,3})(?:$|_)", cn)
    if not m:
        return None
    try:
        return int(m.group("h"))
    except Exception:
        return None

# Use parsed sensor_info if available; otherwise fall back to name parsing
if "sensor_info" in globals() and not sensor_info.empty and "hour" in sensor_info.columns:
    hour_counts = sensor_info["hour"].value_counts().sort_index()
else:
    hours_found = [extract_hour_from_colname(c) for c in sensor_cols]
    hour_counts = pd.Series([h for h in hours_found if h is not None]).value_counts().sort_index()

print("Distinct hour numbers found in sensor column names:", int(hour_counts.shape[0]))
display(hour_counts.head(30))

rows_per_scenario_max = None
if "counts" in globals():
    try:
        rows_per_scenario_max = int(counts.max())
    except Exception:
        rows_per_scenario_max = None

long_hint = (scenario_id_col is not None) and (rows_per_scenario_max is not None) and (rows_per_scenario_max > 1) and (hour_col is not None or timestamp_col is not None)
wide_hint = int(hour_counts.shape[0]) >= 12

print()
print("Rows per scenario (max):", rows_per_scenario_max)
print("LONG hint (multiple rows per scenario + hour/timestamp col):", bool(long_hint))
print("WIDE hint (hours in column names):", bool(wide_hint))

Distinct hour numbers found in sensor column names: 24


hour
0     21
1     21
2     21
3     21
4     21
5     21
6     21
7     21
8     21
9     21
10    21
11    21
12    21
13    21
14    21
15    21
16    21
17    21
18    21
19    21
20    21
21    21
22    21
23    21
Name: count, dtype: int64


Rows per scenario (max): 1
LONG hint (multiple rows per scenario + hour/timestamp col): False
WIDE hint (hours in column names): True


In [17]:
# 24-hour completeness checks (only relevant for LONG format)
if scenario_id_col is None:
    print('No scenario_id detected; skipping per-scenario 24-hour checks.')
elif "counts" in globals() and int(counts.max()) == 1:
    print('Detected WIDE dataset (1 row per scenario); skipping per-scenario 24-hour checks.')
elif hour_col is not None:
    per_scenario_hours = df.groupby(scenario_id_col)[hour_col].nunique(dropna=False)
    print('Unique values per scenario for hour-like column (nunique):')
    display(per_scenario_hours.describe())
    display(per_scenario_hours.value_counts().head(20).rename_axis('unique_values').reset_index(name='num_scenarios'))
elif timestamp_col is not None:
    per_scenario_rows = df.groupby(scenario_id_col).size().rename('rows_per_scenario')
    print('Rows per scenario (timestamp-based):')
    display(per_scenario_rows.describe())
else:
    print('No hour/timestamp detected; cannot verify 24-hour structure.')

Detected WIDE dataset (1 row per scenario); skipping per-scenario 24-hour checks.


In [13]:
# Target columns (explicit)
explicit_targets = [TARGET_X_COL, TARGET_Y_COL, TARGET_BURST_COL]
present_explicit_targets = [c for c in explicit_targets if c in df.columns]
missing_explicit_targets = [c for c in explicit_targets if c not in df.columns]

print('Explicit targets found:', present_explicit_targets)
print('Explicit targets missing:', missing_explicit_targets)
if present_explicit_targets:
    display(df[present_explicit_targets].describe(include='all').T)

# Target candidates by name + excluding sensor-like set (useful if targets are named differently)
sensor_set = set(sensor_cols)
candidate_targets = [c for c in target_name_candidates if c not in sensor_set]
print()
print('Candidate target columns (by name & not sensor-like):', candidate_targets)
if candidate_targets:
    display(df[candidate_targets].describe(include='all').T)

# Also show numeric columns excluding sensor-like columns (often includes targets)
other_numeric = [c for c in numeric_cols if c not in sensor_set]
print()
print('Numeric columns excluding sensor-like columns:', len(other_numeric))
df[other_numeric].describe().T.sort_values('std', ascending=False).head(30)

Explicit targets found: ['leak_x', 'leak_y', 'leak_size_lpm']
Explicit targets missing: []


,count,mean,std,min,25%,50%,75%,max
leak_x,8000.0,3.352648e+05,91.703268,3.349102e+05,3.352012e+05,3.352574e+05,3.353308e+05,3.354884e+05
leak_y,8000.0,3.064730e+06,201.479591,3.064276e+06,3.064618e+06,3.064768e+06,3.064875e+06,3.065107e+06
leak_size_lpm,8000.0,3.153513e+01,48.302840,2.771664e-01,1.648379e+00,5.149952e+00,2.612816e+01,1.629382e+02



Candidate target columns (by name & not sensor-like): ['leak_x', 'leak_y', 'leak_size_lpm']


,count,mean,std,min,25%,50%,75%,max
leak_x,8000.0,3.352648e+05,91.703268,3.349102e+05,3.352012e+05,3.352574e+05,3.353308e+05,3.354884e+05
leak_y,8000.0,3.064730e+06,201.479591,3.064276e+06,3.064618e+06,3.064768e+06,3.064875e+06,3.065107e+06
leak_size_lpm,8000.0,3.153513e+01,48.302840,2.771664e-01,1.648379e+00,5.149952e+00,2.612816e+01,1.629382e+02



Numeric columns excluding sensor-like columns: 13


,count,mean,std,min,25%,50%,75%,max
scenario_id,8001.0,4.000000e+03,2309.834085,0.000000e+00,2.000000e+03,4.000000e+03,6.000000e+03,8.000000e+03
leak_y,8000.0,3.064730e+06,201.479591,3.064276e+06,3.064618e+06,3.064768e+06,3.064875e+06,3.065107e+06
leak_x,8000.0,3.352648e+05,91.703268,3.349102e+05,3.352012e+05,3.352574e+05,3.353308e+05,3.354884e+05
leak_size_lpm,8000.0,3.153513e+01,48.302840,2.771664e-01,1.648379e+00,5.149952e+00,2.612816e+01,1.629382e+02
emitter_coeff_added,8000.0,1.078378e+01,16.715871,1.031000e-01,8.767000e-01,2.136100e+00,1.134565e+01,5.157110e+01
emitter_coeff_total,8000.0,1.078630e+01,16.715870,1.053900e-01,8.790960e-01,2.138602e+00,1.134826e+01,5.157381e+01
leak_start_hr,8000.0,1.138425e+01,6.918839,0.000000e+00,5.000000e+00,1.100000e+01,1.700000e+01,2.300000e+01
leak_duration_hr,8000.0,3.635500e+00,2.736526,1.000000e+00,2.000000e+00,3.000000e+00,5.000000e+00,1.200000e+01
leak_node_pressure,8000.0,2.334038e+01,2.419664,7.176334e-03,2.257524e+01,2.365759e+01,2.468713e+01,2.760996e+01
leak_zone_multiplier,8000.0,9.362883e-01,0.056332,8.501911e-01,8.555080e-01,9.565314e-01,9.859826e-01,1.007545e+00


## Prepare baseline dataset (wide format)
This section **prepares the dataset for the wide-format baseline** by keeping only:
- `scenario_id`
- targets: `leak_x`, `leak_y`, `leak_size_lpm`
- all detected sensor columns like `NODEADD_..._HOUR...`
and dropping everything else.

In [18]:
# Keep only: scenario_id + targets + sensor columns (wide format baseline)
if scenario_id_col is None:
    raise ValueError(
        "No scenario_id column selected. Set SCENARIO_ID_COL to the correct column name in the config cell."
    )

required_targets = [TARGET_X_COL, TARGET_Y_COL, TARGET_BURST_COL]
missing_targets = [c for c in required_targets if c not in df.columns]
if missing_targets:
    print("WARNING: Missing explicit target columns:", missing_targets)

keep_cols = [scenario_id_col]
keep_cols += [c for c in required_targets if c in df.columns]
keep_cols += list(sensor_cols)

# De-dup while preserving order
seen = set()
keep_cols = [c for c in keep_cols if not (c in seen or seen.add(c))]

df_wide_ready = df.loc[:, keep_cols].copy()

print("Prepared wide-format baseline dataset")
print(" - Rows:", int(df_wide_ready.shape[0]))
print(" - Columns:", int(df_wide_ready.shape[1]))
print(" - scenario_id col:", scenario_id_col)
print(" - targets kept:", [c for c in required_targets if c in df_wide_ready.columns])
print(" - sensor cols kept:", int(len([c for c in df_wide_ready.columns if c in set(sensor_cols)])))

# Quick sanity checks
dup_scenarios = int(df_wide_ready[scenario_id_col].duplicated().sum())
if dup_scenarios:
    print(f"NOTE: scenario_id has {dup_scenarios} duplicated rows (dataset may contain multiple rows per scenario).")
else:
    print("scenario_id appears unique per row.")

out_path = (PROJECT_ROOT / "data" / "processed" / "leak_dataset_wide_baseline.csv").resolve()
out_path.parent.mkdir(parents=True, exist_ok=True)
df_wide_ready.to_csv(out_path, index=False)
print("Saved:", out_path.as_posix())

Prepared wide-format baseline dataset
 - Rows: 8001
 - Columns: 508
 - scenario_id col: scenario_id
 - targets kept: ['leak_x', 'leak_y', 'leak_size_lpm']
 - sensor cols kept: 504
scenario_id appears unique per row.
Saved: /home/sujalmainali727/Projects/leak-detection-models/data/processed/leak_dataset_wide_baseline.csv


In [19]:
@dataclass
class DatasetMetadata:
    csv_path: str
    shape: Tuple[int, int]
    columns: List[str]
    dtypes: Dict[str, str]
    missing_by_column: Dict[str, int]
    detected: Dict[str, Any]


sensor_summary: Dict[str, Any] = {"sensor_cols_count": int(len(sensor_cols))}
if "sensor_info" in globals() and not sensor_info.empty and {"node_type", "node_id", "hour"}.issubset(sensor_info.columns):
    # Note: your dataset uses hours 0-23 (per your current config)
    sensor_summary.update(
        {
            "node_types": sensor_info["node_type"].value_counts().to_dict(),
            "unique_nodes": int(sensor_info[["node_type", "node_id"]].drop_duplicates().shape[0]),
            "hour_min": int(sensor_info["hour"].min()),
            "hour_max": int(sensor_info["hour"].max()),
            "hour_distinct": int(sensor_info["hour"].nunique()),
            "missing_hours_0_23": sorted(set(range(0, 24)) - set(sensor_info["hour"].unique().tolist())),
        }
    )

metadata = DatasetMetadata(
    csv_path=csv_path.as_posix(),
    shape=(int(df.shape[0]), int(df.shape[1])),
    columns=list(map(str, df.columns)),
    dtypes={str(k): str(v) for k, v in df.dtypes.astype(str).items()},
    missing_by_column={str(k): int(v) for k, v in df.isna().sum().items()},
    detected={
        "scenario_id": {
            "explicit": SCENARIO_ID_COL if "SCENARIO_ID_COL" in globals() else None,
            "chosen": scenario_id_col,
            "present": bool(scenario_id_col in df.columns) if scenario_id_col is not None else False,
        },
        "targets": {
            "x": TARGET_X_COL,
            "y": TARGET_Y_COL,
            "burst": TARGET_BURST_COL,
        },
        "targets_present": {
            TARGET_X_COL: bool(TARGET_X_COL in df.columns),
            TARGET_Y_COL: bool(TARGET_Y_COL in df.columns),
            TARGET_BURST_COL: bool(TARGET_BURST_COL in df.columns),
        },
        "hour_col": hour_col,
        "timestamp_col": timestamp_col,
        "sensor_prefix_hint": SENSOR_PREFIX_HINT,
        "sensor_node_prefixes": SENSOR_NODE_PREFIXES,
        "sensor_cols_sample": list(map(str, sensor_cols[:100])),
        "sensor_summary": sensor_summary,
        "target_name_candidates": list(map(str, target_name_candidates)),
        "candidate_targets_excluding_sensor": list(map(str, candidate_targets)),
        "duplicate_rows": int(df.duplicated().sum()),
        "hours_in_sensor_colnames_counts": {int(k): int(v) for k, v in hour_counts.items()},
    },
)

out_dir = (PROJECT_ROOT / "outputs" / "metadata").resolve()
out_dir.mkdir(parents=True, exist_ok=True)
out_json = out_dir / "dataset_metadata.json"
out_cols = out_dir / "dataset_columns.csv"

out_json.write_text(json.dumps(asdict(metadata), indent=2), encoding="utf-8")
pd.Series(df.columns, name="column").to_csv(out_cols, index=False)

print("Wrote:")
print(" -", out_json.as_posix())
print(" -", out_cols.as_posix())

Wrote:
 - /home/sujalmainali727/Projects/leak-detection-models/outputs/metadata/dataset_metadata.json
 - /home/sujalmainali727/Projects/leak-detection-models/outputs/metadata/dataset_columns.csv


## Next decisions (fill in after running)
- Confirm final `scenario_id` column name (or confirm data is wide with 1 row per scenario).
- Confirm long vs wide format.
- Confirm sensor column naming scheme (prefix and hour encoding, if any).
- Confirm target columns for `x`, `y`, and `burst_size`.